In [1]:
# Gather links with unstructured
from unstructured.partition.html import partition_html
cnn_lite_url = "https://lite.cnn.com/"
elements = partition_html(url=cnn_lite_url)

links = []

for element in elements:
    if element.metadata.link_urls:
        relative_link = element.metadata.link_urls[0][1:]
        if relative_link.startswith("2024"):
            links.append(f"{cnn_lite_url}{relative_link}")

print(f"We retrieved {len(links)} links to documents from {cnn_lite_url}")

We retrieved 94 links to documents from https://lite.cnn.com/


In [2]:
# Ingest individual articles with Langchain UnstructuredURLLoader
from langchain.document_loaders import UnstructuredURLLoader
loaders = UnstructuredURLLoader(urls=links[:200], show_progress_bar=True)

docs = loaders.load()
print(f"We retrieved {len(docs)} links to documents from {cnn_lite_url}")

# Add id to each document to retrieve it if needed
for i in range(len(docs)):
    docs[i].id = i
    docs[i].metadata['rewritten'] = False

100%|██████████| 94/94 [00:22<00:00,  4.15it/s]

We retrieved 94 links to documents from https://lite.cnn.com/


In [3]:
from dotenv import load_dotenv
load_dotenv()

from os import environ
credentials = {
    "url_LLM": environ.get("WATSONX_URL_LLM"),
    "url_LANGCHAIN": environ.get("WATSONX_URL_LANGCHAIN"),
    "apikey": environ.get("WATSONX_API_KEY")
}
project_id = environ.get("PRJ_ID")

from hap_utilities import clean_hap_content
clean_hap_content(docs, credentials, project_id)

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/mrinalduzzi/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
/Users/mrinalduzzi/Desktop/Projects/RAG-with-watsonx/HAP/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


(86, 66)
Probability: 0.9199702739715576, Sentence: the front for next season: from the flowery, frilly and even a return of the pussy bow at Prada, to
There are 1 documents with at east one sentence with HAP probability > 90%. [86].
Indices of matching strings: [56]
Doc: 86 - chunk: 56 original : Normcore banished the blouse to the back of the closet, but she’s made her way to the front for next season: from the flowery, frilly and even a return of the pussy bow at Prada, to buttoned-up Peter Pan collars at Jil Sander.
Doc: 86 - chunk: 56 rewritten: Next season, Normcore will banish the blouse to the back of the closet, but she's made her way to the front: from the flowery, frilly and even a return of the pussy bow at Prada, to buttoned-up Peter Pan collars at Jil Sander.


## Generate embeddings for the extracted articles

Before loading our knowledge base into a suitable vector DB, we need to generate embeddings for our articles.
Embeddings are a type of transformation that helps computers understand the meaning of words and phrases in a text by converting them into a continuous vector space. This makes it easier for computers to learn and make sense of complex relationships between different concepts.

In generative AI pipelines, embeddings are essential because they allow models to capture the semantic meaning of text data. By mapping words and phrases to vectors, embeddings help models maintain context across sentences and documents, making it possible to generate new text that is both coherent and relevant.

To generate embeddings for our articles, we use IBM_SLATE_30M_ENG model from watsonx.ai model library.

N.B. In order to use IBM Slate model within watsonx, you need to set up 3 environment variables, related to your watsonx.ai instance:
- <b>WATSONX_URL</b>: it is a URL with this format "https://{region}.ml.cloud.ibm.com"
- <b>WATSONX_API_KEY</b>: API KEY from your IBM Cloud account. A detailed procedure on how to create an API KEY can be found in the link provided at the end of this cell.
- <b>PRJ_ID</b>: it is the ID of the project created on watsonx.ai platform to run this notebook

Information on how to find/create these variables can be found here: https://dataplatform.cloud.ibm.com/docs/content/wsj/analyze-data/fm-credentials.html?context=wx&audience=wdp.

In [4]:
from langchain_ibm import WatsonxEmbeddings
from ibm_watsonx_ai.foundation_models.utils.enums import EmbeddingTypes
embeddings = WatsonxEmbeddings(
    model_id=EmbeddingTypes.IBM_SLATE_30M_ENG.value,
    url=credentials["url_LANGCHAIN"],
    apikey=credentials["apikey"],
    project_id=project_id
    )

## Load documents into ChromaDB

With the documents preprocessed and vectorized, we're now ready to load them into ChromaDB. We easily accomplish that leveraging the Chroma integration within Langchain. Once the documents are in Chroma, we can perform a similarity search to retrieve documents related to our topic of interest. Here we choose to limit the retrived documents to 5.

In [5]:
# Split the documents into chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=200, add_start_index=True
)
texts = text_splitter.split_documents(docs)

from langchain_chroma import Chroma
vector_store = Chroma.from_documents(texts, embeddings)
# N.B. The following numbers should be the same
print("Number of documents added to the db:", len(texts))
print("Number of documents within the db:", len(vector_store.get()['documents']))


Number of documents added to the db: 662
Number of documents within the db: 662


We are now ready to receive a question form the user and store it the variable named text

In [6]:
text = "Who is Kim Jones?"

We now perform a first similarity search and retrive the 10 most relevant articles and print them out. We expect this search to return documents that are not directly related to the question that the user proposed to the pipeline. We are only sorting the returned documents based on their  distance.

In [7]:
def pretty_print_docs(docs):
    print(
        f"\n{'-' * 100}\n".join(
            [f"Document {i+1}: \n\n" + d.page_content for i, d in enumerate(docs)]
        )
    )
    
query_docs = vector_store.similarity_search(text, k=10)

pretty_print_docs(query_docs)

Document 1: 

Perfectly imperfect 

High octane glamour was dialled right down, too. At Fendi, Kim Jones presented a 1920s-themed collection anticipating the brand’s centenary next year. The delicate Art Deco dropped-waist frocks and sheer tea dresses were nearly all styled with chunky lace-up boots by Red Wing. “I didn’t want it to be too ladylike when you have these dresses which are very ‘20s looking, I wanted to make it more into the girl that I know,” he said backstage, name-checking close friend Kate Moss as his high-low muse. “When you’re on holiday (and) you’re watching Kate get dressed for dinner, it’s quite fun to see the amount of different clothes getting tried on just to go for dinner on the beach.”
----------------------------------------------------------------------------------------------------
Document 2: 

Keough is left to mourn, along with her sisters, Finley Aaron Love Lockwood and Harper Vivienne Ann Lockwood. Their brother, Benjamin Keough, died at the age of 27

## Compress and Summarize the Documents

After retrieving relevant documents from Chroma, we're ready to compress and then summarize them! There are multiple ways to accomplish this in `langchain`, but `ContextualCompressionRetriever` and `load_summarization_chain` is quite straightforward. In this case we're going to use the IBM Granite model within the available Langchain wrapper so to easily integrate it with our summarization chain. Here we limit the summary to snippets related to our topic of choice.

In order to use IBM Granite model within watsonx, we will use the environment variables previously set in the notebook.

In [8]:
from ibm_watsonx_ai.metanames import GenTextParamsMetaNames as GenParams
from ibm_watsonx_ai.foundation_models.utils.enums import DecodingMethods
from langchain_ibm import WatsonxLLM

granite = WatsonxLLM(
    model_id='ibm/granite-13b-chat-v2',
    url=credentials["url_LANGCHAIN"],
    apikey=credentials["apikey"],
    project_id=project_id,
    params= {
        GenParams.DECODING_METHOD: DecodingMethods.SAMPLE.value,
        GenParams.MAX_NEW_TOKENS: 1024,
        GenParams.MIN_NEW_TOKENS: 1,
        GenParams.TEMPERATURE: 0.5,
        GenParams.TOP_K: 50,
        GenParams.TOP_P: 1
    }
)

Here we compress the found documents by leveraging the LLM again before passing them into the chain for geneating the answer

In [9]:
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import LLMChainExtractor   

compressor = LLMChainExtractor.from_llm(granite)
compression_retriever = ContextualCompressionRetriever(base_compressor=compressor, base_retriever=vector_store.as_retriever())

compressed_docs = compression_retriever.invoke(text)
pretty_print_docs(compressed_docs)

Document 1: 

* Kim Jones is a fashion designer.
* He presented a collection for Fendi.
* The collection was themed around the 1920s.
* The collection featured chunky lace-up boots by Red Wing.
* Jones mentioned Kate Moss as his high-low muse.
* Jones is friends with Kate Moss.

No output, as the context does not provide any information about Kim Jones's nationality or birthplace.
----------------------------------------------------------------------------------------------------
Document 2: 

* Kim Jones, referred to as "the best mother, a wild child, a fierce friend, an underrated artist, frank, funny, traumatized, joyous, grieving, everything that she was throughout her remarkable life" is the daughter of Benjamin Keough and actress Lisa Marie Presley.

No output, as the context does not provide any information about Kim Jones beyond her familial relationships and her mother's career.
---------------------------------------------------------------------------------------------------

Now we need to set up our summarization chain: the basic chain types are either "stuff" (i.e. documents are provided as context in a single prompt that is passed to the LLM) or "map-reduce" (i.e. documents are processed in a map-reduce fashion in order to obtain summaries from each single document and provide these summaries as context to the LLM). 

In order to avoid possible limits in the number of token processed by the LLM, we could opt for a map-reduce chain (more information on summarization in langchain can be found at https://python.langchain.com/docs/use_cases/summarization) since we applied compression we can stick with the "stuff" type.

We also define a prompt to pass down the invocation along witht he resulting input documents from the retriever and the compressor.

In [10]:
from langchain.chains.summarize import load_summarize_chain
chain = load_summarize_chain(granite, chain_type="stuff")

input = {
    "prompt" : "You are a AI language model designed to function as a specialized Retrieval Augmented Generation (RAG) assistant. When generating responses, prioritize correctness, i.e., ensure that your response is correct given the context and user query, and that it is grounded in the context. Furthermore, make sure that the response is supported by the given document or context. When the question cannot be answered using the context or document, output the following response: 'I'm sorry, I don't know.' Always make sure that your response is relevant to the question. If an explanation is needed, first provide the explanation or reasoning, and then give the final answer.",
    "input_documents" : compressed_docs
}
print(chain.invoke(input)['output_text'])



* Kim Jones is a fashion designer who presented a collection for Fendi, themed around the 1920s, featuring chunky lace-up boots by Red Wing.
* Jones is friends with Kate Moss and has mentioned her as his high-low muse.
* No information about Kim Jones's nationality or birthplace is provided.
* Kim Jones is the daughter of Benjamin Keough and actress Lisa Marie Presley.
* No information about Kim Jones beyond her familial relationships and her mother's career is provided.
* The context highlights the issue of potentially putting an innocent person to death and mentions Kim Jones as "Williams" in the context of a capital punishment case.
* Larry Komp is mentioned as one of the attorneys for "Williams".
* No relevant parts were found in the context regarding Kim Jones's nationality or birthplace.


Check out IBM _<a href="https://ibm.github.io/watsonx-ai-python-sdk/samples.html" target="_blank" rel="noopener no referrer">Online Documentation</a>_ for more samples, tutorials, documentation, how-tos, and blog posts.